In [ ]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic # Install core packages for agentic AI and LangChain ecosystem

In [ ]:
# Install LiteLLM and upgrade CrewAI
%pip install litellm 
%pip install -U crewai

In [ ]:
import os #Import the os module to access environment variables
import requests #Import the requests module to make HTTP requests
import litellm #Import the litellm module to interact with LiteLLM
from crewai.llm import LLM #Import the LLM class from the crewai.llm module to interact with language models
from crewai import Agent, Task, Crew #Import the Agent, Task, and Crew classes from the crewai module to create and manage agents and tasks
from crewai.tools import BaseTool #Import the BaseTool class from the crewai.tools module to create custom tools for agents
from dotenv import load_dotenv #Import the load_dotenv function from the dotenv module to load environment variables from a .env file

load_dotenv()

# Tells LiteLLM to silently drop any parameters a given model/provider doesn't support, instead of throwing an error. This adds robustness when switching between providers with different parameter sets.
litellm.drop_params = True 

# Monkey patch litellm.completion to handle parameter mapping
# This is a monkey patch — it overwrites litellm.completion with a 
# wrapper function at runtime. Newer OpenAI/Azure models 
# (like GPT-5 family, per the PDF) expect the parameter 
# max_completion_tokens instead of the older max_tokens. 
# Since CrewAI's LLM class internally passes max_tokens, 
# this patch intercepts every call and renames that 
# parameter automatically before it reaches the 
# API — avoiding an API error without having to rewrite 
# CrewAI's internals. This is a fix in order to run it locally 
# with using GPT-5-mini LLM
original_completion = litellm.completion

def patched_completion(*args, **kwargs):
    # If max_tokens is present and max_completion_tokens is not, map it
    if 'max_tokens' in kwargs and 'max_completion_tokens' not in kwargs:
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens')
    
    return original_completion(*args, **kwargs)

# Apply the patch
litellm.completion = patched_completion

In [ ]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# ---- Custom CrewAI Tool for Web Search ----

class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    # CrewAI actual call to the tool,  
    # It POSTs the query to Tavily's search API, requests up to 
    # 3 results, and formats each result as "Title - URL" 
    # joined by newlines.
    def _run(self, query: str): 
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)

search_tool = TavilySearchTool()

# ---- Azure LLM - FIXED ----
# Using max_tokens (not max_completion_tokens) with the monkey patch
llm = LLM(
    # the 'azure/' prefix tells LiteLLM which provider adapter to use; 
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    is_litellm=True, #tells CrewAI to route calls through LiteLLM rather than a native SDK.
    temperature=1, # some models don't support temperature, so set it to 1 to avoid errors. 1 means full randomness/creativity in responses
    max_tokens=3500  # this is the parameter that gets caught by the monkey patch defined above and silently converted to max_completion_tokens=3500 before being sent to Azure. it causes error when directly use 'max_completion_tokens' param here
)

In [ ]:
# Researcher agent
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True, #prints the agent's reasoning/actions to the console as it works.
    allow_delegation=False, # prevents this agent from handing off subtasks to other agents.
    llm=llm, #wires in the Azure model defined above.
    max_iter=1, # caps the agent to a single reasoning loop/tool-call cycle before it must produce a final answer — keeps it fast and prevents excessive back-and-forth tool calling.
    tools=[search_tool] # gives it access to the Tavily web search tool.
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

In [ ]:
# ---- Tasks ----
# Defines the research task and assigns it to the researcher agent. 
# 'description' is the instruction; 
# 'expected_output' tells the agent (and downstream consumers) what shape the answer should take.

task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

# Defines the writing task, assigned to writer. 
# The key part is context=[task_research] — this tells CrewAI to feed the output of task_research into this task automatically, 
# so the writer has the researcher's notes available without you manually passing data between them.
task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 100 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]
)

In [ ]:
# ---- Crew ----

# Since tasks is a list and no process is specified, CrewAI defaults to sequential process — task_research runs first, 
# then task_write runs using its output.
crew = Crew( # bundles the agents and tasks together. 
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

# starts execution asynchronously (hence await, 
# which only works in a notebook cell or async context — 
# this is why it's in a .ipynb rather than a plain script).
result = await crew.kickoff_async()

print("\nFinal Output:\n")
print(result.raw) # Is the final task's raw text output (the writer's executive summary), which gets printed.